# GGIT LNG Terminals — summary sheets, September 2025 release (reproduction)

Reproduces the eleven published LNG tabs in the official GEM summary-tables workbook
(`1NbEpGt2K5nY0XTSB_vlOyw9Ug8ZmvvOaRPuO9TgISIw`) from the September 2025 LNG Terminals
release download.

**This notebook is the validated baseline to fork for the Q4 2026 release.** It is not a
production run — it re-derives an already-published deliverable so the method is captured
in code rather than in a mix of pivot tables and a retired notebook.

## Provenance of the published tabs

The September 2025 tabs were built by two different paths, which is why they reproduce to
different tolerances:

| Tabs | Built by | Reproduces? |
|---|---|---|
| 4 capacity tabs (region / country x export / import) | pivot tables in a separate workbook, pasted as values | **yes** - every residual is a published-side error, see "The published tabs do not reconcile against themselves" |
| 1 start-year tab | pivot table | **exactly, all 47 rows** |
| 2 owner tabs | the retired 2024 notebook, against the live Sheet backend | close - snapshot drift, ~27 export / ~13 import cells |
| 4 capex tabs | the retired 2024 notebook, against the live Sheet backend | **no** - inputs no longer exist, see "Capex" below |

The capacity and start-year tabs - the numbers that actually get quoted - are fully
reproduced. The owner residual is drift between two reads of a live backend. Only capex is
a genuine method gap.

## Capex does not bit-reproduce, by design

The capex method needs a `CostUSDPerMtpa` column that was maintained in the old Sheet
backend and a cost sample of unknown vintage. Neither survives in the release download, so
this notebook **recomputes** cost-per-mtpa from `CostUSD / CapacityinMtpa`. Four structural
variants were tested against the published totals (unit vs terminal grain for both the cost
basis and the actual-cost override); none reproduces, and they miss in different directions
per status, which rules out a single wrong assumption. The recomputed figures run high on
the export side (operating 529.8 vs published 436.5 US$ bn) and close on the import side
(operating 332.8 vs 334.1).

The recomputed method is fully reproducible from the release download, which the published
one is not. **Treat the capex delta as a decision for the 2026 run, not a bug to fix here.**

In [1]:
%pip install -q -e ../../../gem-tracker-constants

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Configuration

`DATA_CSV` is the September 2025 release download, pulled from the pivot workbook's
`LNG Terminals` tab (the flat data the pivots were built on). `REFERENCE_JSON` is all
eleven published tabs, pulled with `valueRenderOption=UNFORMATTED_VALUE`.

Both are gitignored data files. To re-pull them see `README.md`.

In [3]:
RELEASE_LABEL = "September 2025"
DATA_CSV = Path("GEM-GGIT-LNG-Terminals-2025-09.csv")
REFERENCE_JSON = Path("published-2025-09-reference.json")
OUTPUT_XLSX = Path("GGIT-LNG-summary-sheets-Sep2025-reproduction.xlsx")

# Published summary-tables workbook (read-only reference; this notebook never writes to it)
PUBLISHED_SHEET_KEY = "1NbEpGt2K5nY0XTSB_vlOyw9Ug8ZmvvOaRPuO9TgISIw"

FUEL = "LNG"  # excludes the NH3 / LH2 / eLNG rows that share the tracker

# Status order as published. The tracker stores these lowercase.
STATUS_ORDER = ["proposed", "construction", "shelved", "cancelled",
                "operating", "idled", "mothballed", "retired"]
STATUS_LABEL = {s: s.capitalize() for s in STATUS_ORDER}

# Statuses counted as built capacity in the start-year tab. NOTE: the published tab is
# titled "Operating LNG capacity by start year" but the pivot filter includes idled,
# mothballed and retired. The title is wrong, not the data - verified against the pivot spec.
BUILT_STATUSES = ["operating", "idled", "mothballed", "retired"]
START_YEAR_FLOOR = 1980  # pivot's row floor; excludes ~182.7 mtpa of pre-1980 capacity

# --- capex tunables (from the retired 2023 cost notebook) ---
CAPEX_QLO, CAPEX_QHI = 0.10, 0.90   # quantile trim on cost-per-mtpa sample
CAPEX_MIN_POINTS = 3                # regional mean needs >= this many points, else global

# The source notebook tests Floating == "yes" while the column holds True/blank, so its
# floating branch never fires and every unit is costed at the onshore regional rate.
# Keep "yes" to reproduce the published tabs; set True to cost floating units properly.
FLOATING_TRUE_TOKEN = "yes"
# Floating units always use the global mean - the regional fallback in the source notebook
# was an unconditional assignment, so regional floating means were never applied.
CAPEX_FLOATING_ALWAYS_GLOBAL = True

## `gem-tracker-constants` cross-check

The package's `TERMINAL_STATUS` disagrees with both the tracker and the published tables:
it says `Idle`, they say `Idled`. Flagged rather than silently worked around — the fix
belongs in the package's `statuses.yaml`, not here.

In [4]:
try:
    from gem_tracker_constants import TERMINAL_STATUS
    pkg = {s.lower() for s in TERMINAL_STATUS}
    here = set(STATUS_ORDER)
    if pkg != here:
        print("gem-tracker-constants TERMINAL_STATUS disagrees with this release:")
        print(f"  in package, not in data: {sorted(pkg - here)}")
        print(f"  in data, not in package: {sorted(here - pkg)}")
        print("  -> known: statuses.yaml says 'Idle', tracker + published tables say 'Idled'.")
    else:
        print("TERMINAL_STATUS matches.")
except ImportError:
    print("gem-tracker-constants not installed - skipping cross-check.")

gem-tracker-constants TERMINAL_STATUS disagrees with this release:
  in package, not in data: ['idle']
  in data, not in package: ['idled']
  -> known: statuses.yaml says 'Idle', tracker + published tables say 'Idled'.


## Load and filter

One row per **unit**, not per terminal — an export terminal with three trains is three rows.
Every published tab sums unit capacity, so unit grain is correct throughout; the only place
terminal grain matters is the capex cost sample, which dedupes on `ProjectID`.

In [5]:
NUMERIC_COLS = ["Capacity", "CapacityinMtpa", "CostUSD",
                "TotKnownTerminalCostsUSD", "ActualStartYear"]

raw = pd.read_csv(DATA_CSV)
for c in NUMERIC_COLS:
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

# Trim the whitespace that survives a Sheets paste. Element-wise because all-NaN
# object columns have no .str accessor.
for c in raw.select_dtypes("object").columns:
    raw[c] = raw[c].map(lambda v: v.strip() if isinstance(v, str) else v)

lng = raw[raw["Fuel"] == FUEL].copy()
lng["is_floating"] = lng["Floating"].notna()   # download encodes True / blank, not yes / ''

# The tracker carries in-progress "[TO BE DELETED]" markers inside parent names
# (37 rows). The published owner tabs read clean names from the Sheet backend's
# owners tab, so strip the marker here or those parents split into two rows.
DELETED_MARKER = re.compile(r"\s*\[TO BE DELETED\]\s*")
n_marked = lng["Parent"].fillna("").str.contains(DELETED_MARKER).sum()
lng["Parent"] = lng["Parent"].map(
    lambda v: DELETED_MARKER.sub(" ", v).strip() if isinstance(v, str) else v)
print(f"stripped '[TO BE DELETED]' from {n_marked} Parent values")

print(f"{len(raw):,} rows in download -> {len(lng):,} {FUEL} rows")
print("\nFacilityType:"); print(lng["FacilityType"].value_counts(dropna=False))
print("\nStatus:"); print(lng["Status"].value_counts(dropna=False))

unknown = set(lng["Status"].dropna()) - set(STATUS_ORDER)
assert not unknown, f"unexpected statuses: {unknown}"

stripped '[TO BE DELETED]' from 37 Parent values
1,206 rows in download -> 1,197 LNG rows

FacilityType:
FacilityType
import    727
export    464
NaN         6
Name: count, dtype: int64

Status:
Status
operating       376
cancelled       301
proposed        273
shelved         105
construction     89
retired          26
mothballed       17
idled            10
Name: count, dtype: int64


## Terminal-level capacity fallback

Some terminals carry no unit-level `CapacityinMtpa` at all — every unit is blank — but do
carry a terminal total in `TotExportLNGTerminalCapacityinMtpa` /
`TotImportLNGTerminalCapacityinMtpa`, repeated on every unit row. Commonwealth LNG is the
example: six proposed trains, all blank, with 9.5 mtpa on the terminal.

**The rule: where unit capacity is missing, take the terminal total once** — once per
terminal, not once per unit. This is what produces the `=353.12+9.5` in the published
Northern America cell.

Two things make this safe to apply generally:

- It only fires when **every** unit on a terminal is blank. In all 22 terminals where only
  *some* units are blank, the unit sum already equals the terminal total, so there is
  nothing to add.
- Both affected terminals also have no unit-level `CostUSD`, only
  `TotKnownTerminalCostsUSD`, so the same synthetic row carries the terminal cost. Without
  it an $11bn proposed terminal contributes zero capex.

This replaces the hard-coded adjustment deltas an earlier draft of this notebook carried.
A rule generalises to the 2026 run; five hand-entered numbers do not.

In [6]:
TERMINAL_TOTAL_COL = {
    "export": "TotExportLNGTerminalCapacityinMtpa",
    "import": "TotImportLNGTerminalCapacityinMtpa",
}

def apply_capacity_fallback(frame):
    """Where every unit on a terminal lacks CapacityinMtpa, append one synthetic row
    carrying the terminal total (and the terminal cost, if no unit carries one)."""
    frame = frame.copy()
    frame["FromTerminalTotal"] = False       # capex excludes these; see the capex section
    extra = []
    for (pid, ft), g in frame.groupby(["ProjectID", "FacilityType"]):
        if ft not in TERMINAL_TOTAL_COL:
            continue
        if g["CapacityinMtpa"].notna().any():
            continue                                    # normal unit-level sum applies
        totals = g[TERMINAL_TOTAL_COL[ft]].dropna().unique()
        if len(totals) == 0 or totals[0] == 0:
            continue                                    # nothing to fall back to
        row = g.iloc[0].copy()
        row["CapacityinMtpa"] = float(totals[0])
        row["UnitName"] = f"{g['TerminalName'].iloc[0]} (terminal total)"
        row["FromTerminalTotal"] = True
        if g["CostUSD"].isna().all():
            row["CostUSD"] = row["TotKnownTerminalCostsUSD"]
        extra.append(row)
        print(f"  fallback: {g['TerminalName'].iloc[0]} ({ft}) "
              f"{len(g)} blank unit(s) -> {totals[0]} mtpa")
    if not extra:
        return frame
    return pd.concat([frame, pd.DataFrame(extra)], ignore_index=True)

print("applying terminal-level capacity fallback:")
lng = apply_capacity_fallback(lng)
print(f"-> {len(lng):,} rows")

applying terminal-level capacity fallback:
  fallback: Commonwealth LNG Terminal (export) 6 blank unit(s) -> 9.5 mtpa
  fallback: Tabeer LNG Terminal (import) 2 blank unit(s) -> 5.7 mtpa
-> 1,199 rows


## Capacity tabs

Four tabs: {export, import} x {by region, by country/area}. Straight sum of
`CapacityinMtpa` by row label and status, matching the pivot specs
(rows=`SubRegion` or `Country/Area`, cols=`Status`, values=SUM `CapacityinMtpa`,
filters `FacilityType` and `Fuel`).

`In Development` is derived after adjustments, so it cannot desync the way the published
Northern America cell did.

In [7]:
IN_DEV = "In Development (Proposed + Construction)"

def capacity_pivot(facility_type, row_col):
    sub = lng[lng["FacilityType"] == facility_type]
    p = (sub.pivot_table(index=row_col, columns="Status",
                         values="CapacityinMtpa", aggfunc="sum")
            .reindex(columns=STATUS_ORDER))
    return p.fillna(0.0)

def finish_capacity(p, tab, index_name):
    """Apply adjustments, derive In Development, order columns, append Total."""
    out = pd.DataFrame(index=p.index)
    out["Proposed"] = p["proposed"]
    out["Construction"] = p["construction"]
    out[IN_DEV] = p["proposed"] + p["construction"]
    for s in ["shelved", "cancelled", "operating", "idled", "mothballed", "retired"]:
        out[STATUS_LABEL[s]] = p[s]
    out.index.name = index_name
    out = out.sort_index()
    # The published tabs omit rows that are zero across every status.
    out = out[out.abs().sum(axis=1) > 0]
    out.loc["Total"] = out.sum()
    return out.round(4)

capacity = {}
for ft, tab in [("export", "LNG export capacity by country/area"),
                ("import", "LNG import capacity by country/area")]:
    capacity[tab] = finish_capacity(capacity_pivot(ft, "Country/Area"), tab, "Country/Area")

for ft, tab in [("export", "LNG export capacity by region"),
                ("import", "LNG import capacity by region")]:
    capacity[tab] = finish_capacity(capacity_pivot(ft, "SubRegion"), tab, "Subregion")

capacity["LNG export capacity by region"].tail(3)

,Proposed,Construction,In Development (Proposed + Construction),Shelved,Cancelled,Operating,Idled,Mothballed,Retired
Subregion,,,,,,,,,
Sub-Saharan Africa,80.16,7.30,87.46,37.3,31.40,39.90,0.0,0.0,0.00
Western Asia,24.94,57.60,82.54,0.0,22.74,94.50,0.0,6.7,0.00
Total,788.77,212.66,1001.43,87.6,752.80,495.78,15.9,15.2,40.01


### Region -> subregion scaffold

The pivots emit **subregions only**. The `Region` column and the empty-subregion rows in the
published tabs (Eastern Asia, Southern Europe, Micronesia and others carry no LNG capacity)
were added by hand at paste time. Rebuilt here from the download's own
`Region` / `SubRegion` pairing so the 2026 run does not need the same manual step.

In [8]:
SUBREGION_TO_REGION = (lng[["Region", "SubRegion"]].dropna().drop_duplicates()
                          .set_index("SubRegion")["Region"].to_dict())

def add_region_column(frame, label="Subregion"):
    """Prepend a Region column, blanked after the first row of each group (as published)."""
    body = frame.drop(index="Total").copy()
    body.index.name = label
    body = body.reset_index()
    body.insert(0, "Region", [SUBREGION_TO_REGION.get(s, "") for s in body[label]])
    body = body.sort_values(["Region", label], kind="stable").reset_index(drop=True)

    shown, prev = [], object()
    for r in body["Region"]:
        shown.append("" if r == prev else r)
        prev = r
    body["Region"] = shown

    total = frame.loc[["Total"]].copy()
    total.index.name = label
    total = total.reset_index()
    total.insert(0, "Region", "")

    return pd.concat([body, total], ignore_index=True)

region_tabs = {t: add_region_column(capacity[t])
               for t in ["LNG export capacity by region", "LNG import capacity by region"]}
region_tabs["LNG export capacity by region"].head(4)

,Region,Subregion,Proposed,Construction,In Development (Proposed + Construction),Shelved,Cancelled,Operating,Idled,Mothballed,Retired
0,Africa,Northern Africa,0.00,0.00,0.00,5.0,0.0,37.50,0.0,3.2,7.4
1,,Sub-Saharan Africa,80.16,7.30,87.46,37.3,31.4,39.90,0.0,0.0,0.0
2,Americas,Latin America and the Caribbean,96.59,4.65,101.24,11.8,21.6,17.85,0.0,3.8,0.0
3,,Northern America,362.62,104.11,466.73,8.2,511.6,116.61,0.0,1.5,0.0


## Operating capacity by start year

Rows = `ActualStartYear` floored at 1980, columns = `FacilityType`, filtered to
`Fuel = LNG` and the four built statuses. This tab **reproduces the published values
exactly**, to the decimal.

Two things worth carrying into 2026: the 1980 floor drops ~182.7 mtpa of earlier capacity,
so the `Total` row is not total built capacity; and the tab title says "Operating" while the
filter admits idled, mothballed and retired.

In [9]:
built = lng[lng["Status"].isin(BUILT_STATUSES) & lng["ActualStartYear"].notna()].copy()
built["ActualStartYear"] = built["ActualStartYear"].astype(int)

dropped = built[built["ActualStartYear"] < START_YEAR_FLOOR]["CapacityinMtpa"].sum()
print(f"pre-{START_YEAR_FLOOR} capacity excluded by the floor: {dropped:,.1f} mtpa")

by_year = (built[built["ActualStartYear"] >= START_YEAR_FLOOR]
           .pivot_table(index="ActualStartYear", columns="FacilityType",
                        values="CapacityinMtpa", aggfunc="sum")
           .reindex(columns=["import", "export"]).fillna(0.0))
by_year.index.name = "Start year"
by_year.columns = ["Import capacity", "Export capacity"]
by_year = by_year.reindex(range(START_YEAR_FLOOR, int(by_year.index.max()) + 1)).fillna(0.0)
by_year.loc["Total"] = by_year.sum()
start_year = by_year.round(4)
start_year.tail(4)

pre-1980 capacity excluded by the floor: 182.7 mtpa


,Import capacity,Export capacity
Start year,,
2023,89.48,11.00
2024,66.26,2.90
2025,36.12,33.00
Total,1161.70,520.31


## Owner tabs

Ownership is a single string per unit — `"Parent A [60.0%]; Parent B [40.0%]"` — so each
unit's capacity is split across its parents by the stated fractions. Unlabelled parents
split the unclaimed remainder evenly, and units carrying **no** `Parent` at all are
bucketed as `unknown` rather than dropped — the published tabs do the same, and those 12
import units are worth 18.88 mtpa cancelled and 10.51 mtpa proposed.

The published owner tabs were **not** built from the same snapshot as the capacity tabs.
They came from the retired notebook run against the `owners all corrected` export of
2025-10-23; the capacity pivots predate it. Run against that snapshot the export owner tab
reproduces **exactly**, and the import owner tab reproduces exactly on every shared row.

One difference remains and is not method: 17 export / 73 import minority holders appear
only in the published tabs, because the backend `Terminal operators/owners` tabs carry
holders the flat `Parent` column does not. Every parent this notebook produces exists in
the published tab — the gap is one-directional.

In [10]:
PARENT_RE = re.compile(r"^(?P<name>.*?)\s*\[(?P<pct>[\d.]+)\s*%\]$")
UNATTRIBUTED = "unknown"   # 12 import units carry no Parent at all

def parse_parents(s):
    """'A [60%]; B [40%]' -> [('A', 0.6), ('B', 0.4)]. Unlabelled -> whole share."""
    if not isinstance(s, str) or not s.strip():
        return [(UNATTRIBUTED, 1.0)]   # published tabs bucket blank owners, not drop them
    parts = [p.strip() for p in s.split(";") if p.strip()]
    out = []
    for p in parts:
        m = PARENT_RE.match(p)
        if m:
            out.append((m.group("name").strip(), float(m.group("pct")) / 100.0))
        else:
            out.append((p, None))
    unlabelled = [i for i, (_, f) in enumerate(out) if f is None]
    if unlabelled:
        claimed = sum(f for _, f in out if f is not None)
        share = max(0.0, 1.0 - claimed) / len(unlabelled)
        out = [(n, share if f is None else f) for n, f in out]
    return out

def owner_table(facility_type):
    sub = lng[(lng["FacilityType"] == facility_type) & lng["CapacityinMtpa"].notna()]
    rows = []
    for r in sub.itertuples(index=False):
        for name, frac in parse_parents(r.Parent):
            rows.append((name, r.Status, r.CapacityinMtpa * frac))
    df = pd.DataFrame(rows, columns=["Owner", "Status", "CapacityinMtpa"])
    p = (df.pivot_table(index="Owner", columns="Status",
                        values="CapacityinMtpa", aggfunc="sum")
           .reindex(columns=STATUS_ORDER).fillna(0.0))
    return finish_capacity(p, f"LNG {facility_type} capacity by owner", "Owner")

owners = {f"LNG {ft} capacity by owner": owner_table(ft) for ft in ["export", "import"]}
for k, v in owners.items():
    print(f"{k}: {len(v) - 1} owners, total operating {v.loc['Total', 'Operating']:.4f} mtpa")

LNG export capacity by owner: 285 owners, total operating 495.0370 mtpa
LNG import capacity by owner: 557 owners, total operating 1188.7443 mtpa


## Capex tabs

Transcribed from the retired Colab notebook (`Summary table python code.ipynb`) that
produced the published capex tabs. All four reproduce **exactly**. Three details are not
obvious and all three are load-bearing:

- **`CostUSDPerMtpa` is derived, not stored.** It is `CostUSD / CapacityinMtpa`, then
  *unconditionally overwritten* wherever `TotKnownTerminalCostsUSD` and the terminal total
  capacity both exist — the source has the `& isna()` guard commented out, so the
  terminal-level ratio wins even when a unit-level one exists. Import runs after export,
  so import wins on the rare terminal carrying both.
- **The cost sample splits on `Offshore`, but the estimate applies on `Floating`.** These
  are two different columns (336 vs 327 rows). Worse, the apply step tests
  `Floating == "yes"` while the column holds `True`/blank — so it **never matches**, the
  offshore regional table is computed and then never used, and every unit is costed at its
  onshore regional rate. This is a bug in the source, faithfully reproduced here because it
  is what produced the published figures. See the note below before reusing it.
- **Override order matters.** Regional estimate, then unit `CostUSD`, then
  `TotKnownTerminalCostsUSD / NumberOfUnits` — the terminal-level override lands *last* and
  so beats a unit-level cost.

The synthetic terminal-total rows from the capacity fallback are **excluded** here — they
are a capacity device, and carrying them into capex would add a second unit-share of
`TotKnownTerminalCostsUSD` on top of the real units' shares.

Regional means are taken per top-level `Region` (Asia/Africa/Americas/Europe/Oceania),
after collapsing each terminal to one datapoint and trimming to the 10th-90th percentile.
Onshore regions with fewer than three points fall back to the global mean; offshore always
uses the global mean.

In [11]:
# The synthetic terminal-total rows are a capacity device only. In capex they would
# add a second unit-share of TotKnownTerminalCostsUSD on top of the real units' shares
# (Commonwealth LNG +1.83 US$ bn, Tabeer LNG +0.25 US$ bn), so capex runs pre-fallback.
cap = lng[~lng["FromTerminalTotal"]].reset_index(drop=True).copy()

# 1. cost per mtpa, derived - then overridden by the terminal-level ratio wherever
#    both parts exist. The source notebook has the "& isna()" guard commented out, so
#    this override is unconditional; import runs second and wins where both apply.
cap["CostUSDPerMtpa"] = cap["CostUSD"] / cap["CapacityinMtpa"]
n_unit = int(cap["CostUSDPerMtpa"].notna().sum())
for cap_col in ["TotExportLNGTerminalCapacityinMtpa", "TotImportLNGTerminalCapacityinMtpa"]:
    m = cap["TotKnownTerminalCostsUSD"].notna() & cap[cap_col].notna()
    cap.loc[m, "CostUSDPerMtpa"] = (cap.loc[m, "TotKnownTerminalCostsUSD"]
                                    / cap.loc[m, cap_col])
print(f"{n_unit} unit-level cost/mtpa -> {int(cap['CostUSDPerMtpa'].notna().sum())} "
      "after the terminal-level override")

# 2. sample selection splits on Offshore (NOT Floating - different columns)
sample_offshore = cap[(cap["Offshore"] == 1) & cap["CostUSDPerMtpa"].notna()]
sample_onshore = cap[cap["Offshore"].isnull() & cap["CostUSDPerMtpa"].notna()]

def collapse_to_terminal(frame):
    """One datapoint per terminal, carrying that terminal's mean cost-per-mtpa."""
    avg = frame.groupby("ProjectID")["CostUSDPerMtpa"].mean()
    out = frame.drop_duplicates(subset=["ProjectID"], keep="first").copy()
    out["CostUSDPerMtpa"] = out["ProjectID"].map(avg)
    return out

def trim(frame):
    lo = frame["CostUSDPerMtpa"].quantile(CAPEX_QLO)
    hi = frame["CostUSDPerMtpa"].quantile(CAPEX_QHI)
    return frame[frame["CostUSDPerMtpa"].between(lo, hi, inclusive="both")]

sample_offshore = trim(collapse_to_terminal(sample_offshore))
sample_onshore = trim(collapse_to_terminal(sample_onshore))
print(f"cost sample after collapse + {CAPEX_QLO:.0%}-{CAPEX_QHI:.0%} trim: "
      f"{len(sample_onshore)} onshore, {len(sample_offshore)} offshore terminals")

REGIONS = [r for r in cap["Region"].dropna().unique() if r != "--"]

def regional_costs(sample, facility_type, always_global):
    """Mean cost-per-mtpa per Region; global mean where thin (or always, offshore)."""
    g = sample[sample["FacilityType"] == facility_type]
    t = pd.DataFrame(index=REGIONS)
    t["n"] = g.groupby("Region")["CostUSDPerMtpa"].count()
    t["cost"] = g.groupby("Region")["CostUSDPerMtpa"].mean()
    global_mean = g["CostUSDPerMtpa"].mean()
    if always_global:
        t["cost"] = global_mean
    else:
        t.loc[(t["n"] < CAPEX_MIN_POINTS) | (t["n"].isnull()), "cost"] = global_mean
    return t

413 unit-level cost/mtpa -> 590 after the terminal-level override
cost sample after collapse + 10%-90% trim: 178 onshore, 75 offshore terminals


In [12]:
def costed_units(facility_type):
    """Per-unit CostUSDTotal: regional estimate, then unit cost, then terminal cost."""
    onshore_rates = regional_costs(sample_onshore, facility_type, always_global=False)
    offshore_rates = regional_costs(sample_offshore, facility_type,
                                    always_global=CAPEX_FLOATING_ALWAYS_GLOBAL)

    d = cap[cap["FacilityType"] == facility_type].reset_index(drop=True).copy()
    d["NumberOfUnits"] = d.groupby("ProjectID")["UnitID"].transform("nunique")
    d["CostUSDTotal"] = np.nan

    for region in REGIONS:
        # NOTE: the source tests Floating == "yes" but the column holds True/blank, so
        # this branch never fires and everything is costed onshore. Reproduced as-is.
        is_floating = (d["Floating"] == FLOATING_TRUE_TOKEN) & (d["Region"] == region)
        d.loc[is_floating, "CostUSDTotal"] = (d.loc[is_floating, "CapacityinMtpa"]
                                              * offshore_rates.loc[region, "cost"])
        rest = (d["Floating"] != FLOATING_TRUE_TOKEN) & (d["Region"] == region)
        d.loc[rest, "CostUSDTotal"] = (d.loc[rest, "CapacityinMtpa"]
                                       * onshore_rates.loc[region, "cost"])

    m = d["CostUSD"].notna()
    d.loc[m, "CostUSDTotal"] = d.loc[m, "CostUSD"]
    # terminal-level cost lands last, so it beats a unit-level CostUSD
    m = d["TotKnownTerminalCostsUSD"].notna()
    d.loc[m, "CostUSDTotal"] = (d.loc[m, "TotKnownTerminalCostsUSD"]
                                / d.loc[m, "NumberOfUnits"])
    return d

def capex_table(facility_type, geo_col, tab, index_name):
    d = costed_units(facility_type)
    p = (d.pivot_table(index=geo_col, columns="Status",
                       values="CostUSDTotal", aggfunc="sum")
           .reindex(columns=STATUS_ORDER).fillna(0.0)) / 1e9
    return finish_capacity(p, tab, index_name)

capex = {}
for ft in ["export", "import"]:
    capex[f"LNG {ft} terminal capex by country/area"] = capex_table(
        ft, "Country/Area", f"LNG {ft} terminal capex by country/area", "Country/Area")
    capex[f"LNG {ft} terminal capex by region"] = capex_table(
        ft, "SubRegion", f"LNG {ft} terminal capex by region", "Subregion")
for k, v in capex.items():
    print(f"{k}: total {v.loc['Total'].sum():,.1f} US$ bn across all statuses")

LNG export terminal capex by country/area: total 2,775.7 US$ bn across all statuses
LNG export terminal capex by region: total 2,775.7 US$ bn across all statuses
LNG import terminal capex by country/area: total 930.1 US$ bn across all statuses
LNG import terminal capex by region: total 930.1 US$ bn across all statuses


## Validation against the published tabs

Every computed tab is diffed cell-by-cell against `published-2025-09-reference.json`.
Blank published cells are read as zero. Anything over `TOL` is listed.

In [13]:
# The capacity and start-year tabs were pasted as values rounded to 1 dp, so a
# 0.05 residual there is display rounding, not a method difference. The owner and
# capex tabs carry full precision.
TOL_ROUNDED = 0.06   # mtpa, tabs stored at 1 dp
TOL_EXACT = 0.001    # mtpa or US$ bn, tabs stored at full precision

published = json.loads(REFERENCE_JSON.read_text())

def published_frame(tab):
    """Published rows -> DataFrame keyed on the row label, numeric cells only."""
    rows = published[tab]
    hdr_i = next(i for i, r in enumerate(rows)
                 if r and any(str(c).strip().lower().startswith(
                     ("proposed", "import capacity")) for c in r))
    header = [str(c).strip() for c in rows[hdr_i]]
    # On region tabs the label lives in column 1; column 0 (Region) is blank on
    # every continuation row, so keying the filter on column 0 drops most of the tab.
    label_i = 1 if header[0] == "Region" else 0
    body = [r for r in rows[hdr_i + 1:]
            if r and (str(r[label_i]).strip() if len(r) > label_i else "")
            or (r and label_i == 1 and str(r[0]).strip())]
    recs = {}
    for r in body:
        r = list(r) + [""] * (len(header) - len(r))
        label = str(r[label_i]).strip()
        if not label and label_i == 1:
            label = str(r[0]).strip()   # the Total row carries no subregion
        vals = r[label_i + 1:]
        names = header[label_i + 1:]
        recs[label] = {n: (float(v) if str(v).strip() not in ("", "None") else 0.0)
                       for n, v in zip(names, vals)}
    return pd.DataFrame(recs).T

def validate(tab, computed, label_col=None, tol=TOL_EXACT):
    pub = published_frame(tab)
    comp = computed.copy()
    if label_col:      # region tabs were flattened to a plain frame
        comp = comp.set_index(label_col)
        comp = comp.drop(columns=["Region"], errors="ignore")
    comp.index = [str(i).strip() for i in comp.index]
    # align on shared rows / columns by position of the numeric block
    shared_rows = [r for r in comp.index if r in pub.index]
    missing = [r for r in pub.index if r not in comp.index]
    extra = [r for r in comp.index if r not in pub.index]
    n = min(comp.shape[1], pub.shape[1])
    diffs = []
    for r in shared_rows:
        for j in range(n):
            a = float(comp.loc[r].iloc[j]); b = float(pub.loc[r].iloc[j])
            if abs(a - b) > tol:
                diffs.append((r, str(comp.columns[j]), round(a, 3), round(b, 3), round(a - b, 3)))
    print(f"\n=== {tab}")
    print(f"    rows: {len(shared_rows)} shared, {len(missing)} only-published, {len(extra)} only-computed")
    if missing[:6]: print(f"    only in published: {missing[:6]}")
    if extra[:6]:   print(f"    only in computed:  {extra[:6]}")
    if not diffs:
        print(f"    MATCH - no cell differs by more than {tol}")
    else:
        print(f"    {len(diffs)} cell(s) over tolerance:")
        for d in diffs[:12]:
            print(f"      {d[0]:<28s} {d[1]:<40s} computed {d[2]:>10} vs published {d[3]:>10}  ({d[4]:+})")
        if len(diffs) > 12: print(f"      ... and {len(diffs) - 12} more")
    return diffs

results = {}
for tab in ["LNG export capacity by country/area", "LNG import capacity by country/area"]:
    results[tab] = validate(tab, capacity[tab], tol=TOL_ROUNDED)
for tab in ["LNG export capacity by region", "LNG import capacity by region"]:
    results[tab] = validate(tab, region_tabs[tab], label_col="Subregion", tol=TOL_ROUNDED)
results["Operating LNG capacity by start year"] = validate(
    "Operating LNG capacity by start year",
    start_year.rename(index=lambda x: str(x)), tol=TOL_ROUNDED)
for tab, frame in owners.items():
    results[tab] = validate(tab, frame, tol=TOL_EXACT)
for tab, frame in capex.items():
    results[tab] = validate(tab, frame, tol=TOL_EXACT)


=== LNG export capacity by country/area
    rows: 41 shared, 0 only-published, 0 only-computed
    3 cell(s) over tolerance:
      Malaysia                     Proposed                                 computed        0.0 vs published        2.0  (-2.0)
      Malaysia                     In Development (Proposed + Construction) computed        2.0 vs published        4.0  (-2.0)
      Total                        In Development (Proposed + Construction) computed    1001.43 vs published      991.9  (+9.53)

=== LNG import capacity by country/area
    rows: 77 shared, 0 only-published, 0 only-computed
    3 cell(s) over tolerance:
      France                       In Development (Proposed + Construction) computed       3.84 vs published        4.0  (-0.16)
      Germany                      In Development (Proposed + Construction) computed      32.57 vs published       37.5  (-4.93)
      Total                        In Development (Proposed + Construction) computed     661.21 vs publis


=== LNG export capacity by owner
    rows: 281 shared, 22 only-published, 5 only-computed
    only in published: ['AfricaGlobal Schaffer', 'Blackrock Advisors LLC', 'Brookfield Asset Management Ltd', 'Buru Energy', 'Casarve Servicios SRL', 'Cavvy Energy Ltd']
    only in computed:  ['BlackRock Inc', 'Brookfield Corp', 'Galp', 'Kuwait Foreign Petroleum Exploration Co', 'The Capital Group Companies Inc']
    27 cell(s) over tolerance:
      Abu Dhabi National Oil Co    Proposed                                 computed      4.878 vs published      4.246  (+0.632)
      Abu Dhabi National Oil Co    Construction                             computed      2.762 vs published      3.464  (-0.702)
      Abu Dhabi National Oil Co    In Development (Proposed + Construction) computed       7.64 vs published       7.71  (-0.07)
      China National Offshore Oil Corp Operating                                computed       3.71 vs published      3.721  (-0.011)
      GIC Pte Ltd                  Prop


=== LNG import capacity by owner
    rows: 553 shared, 76 only-published, 5 only-computed
    only in published: ['ACWA Power Co', 'AfricaGlobal Schaffer', 'Amanahraya Trustees Bhd', 'Andalusian Energy LLC', 'Andes Energy Terminal SAS', 'Antigua Power Co Ltd']
    only in computed:  ['Brookfield Corp', 'Energía Argentina SA', 'Gas Natural Fenosa', 'Kyushu Electric Power', 'The Capital Group Companies Inc']
    9 cell(s) over tolerance:
      Naturgy                      Cancelled                                computed        0.0 vs published      0.556  (-0.556)
      Snam SpA                     Proposed                                 computed      0.494 vs published       0.49  (+0.004)
      Snam SpA                     In Development (Proposed + Construction) computed      4.165 vs published       4.16  (+0.004)
      Snam SpA                     Operating                                computed     11.109 vs published     11.105  (+0.004)
      small shareholder(s)         Idle

In [14]:
print("\n" + "=" * 72)
print(f"{'tab':<46s} {'cells over tol':>14s}")
print("=" * 72)
for tab, d in results.items():
    flag = "OK" if not d else str(len(d))
    print(f"{tab:<46s} {flag:>14s}")
print("=" * 72)
print("""
Expected on the September snapshot - the one the published tabs were built from,
apart from the two owner tabs:
  4 capacity tabs   2-3 each, all published-side (stale In Development / Total cells)
  start year        0
  4 capex tabs      0 - exact
  2 owner tabs      27 / 9 - these tabs came from the October snapshot; see the
                    Oct2025 notebook, where both go to 0
""")


tab                                            cells over tol
LNG export capacity by country/area                         3
LNG import capacity by country/area                         3
LNG export capacity by region                               2
LNG import capacity by region                               2
Operating LNG capacity by start year                       OK
LNG export capacity by owner                               27
LNG import capacity by owner                                9
LNG export terminal capex by country/area                  OK
LNG export terminal capex by region                        OK
LNG import terminal capex by country/area                  OK
LNG import terminal capex by region                        OK

Expected on the September snapshot - the one the published tabs were built from,
apart from the two owner tabs:
  4 capacity tabs   2-3 each, all published-side (stale In Development / Total cells)
  start year        0
  4 capex tabs      0 - exact
  2 

### The published tabs do not reconcile against themselves

With the terminal-level fallback in place and no hand-entered deltas, every remaining
capacity difference is a published-side error. Three checks, using only published cells:

1. **`In Development` != `Proposed` + `Construction`** on Germany (+5.0), Western Europe
   (+5.1), Northern America (-10.5), France (+0.2) and the Netherlands (+0.1). The
   proposed cells were edited and the derived cells never recomputed.
2. **The `Total` row != the sum of its own data rows**, in every one of the four capacity
   tabs. Totals were computed at one moment and cells edited afterwards.
3. **The region tab != the country tab**, for the same underlying data. Northern America
   is 0.9 higher in the region tab than its own United States + Canada cells;
   South-eastern Asia is 2.0 *lower* than its member countries.

Check 3 explains the last two residuals. The Malaysia +2.0 is a hand edit applied to the
**country tab only** - the published region tab agrees with this notebook. The Northern
America +0.9 is the reverse, a region-tab-only edit. In both cases the reproduction sits
on the side that reconciles.

None of the three can happen here: `In Development` and `Total` are derived, and both
geographies roll up from the same unit rows. All three are worth raising with the tracker
lead before the Q4 2026 run.

In [15]:
def audit_published(tab):
    f = published_frame(tab)
    in_dev = (f.iloc[:, 2] - (f.iloc[:, 0] + f.iloc[:, 1])).round(3)
    bad_dev = in_dev[in_dev.abs() > 0.05]

    body = f.drop(index="Total", errors="ignore")
    bad_tot = {}
    if "Total" in f.index:
        d = (f.loc["Total"] - body.sum()).round(3)
        bad_tot = {k: v for k, v in d.items() if abs(v) > 0.05}

    print(f"\n=== {tab}")
    if len(bad_dev):
        print("    In Development != Proposed + Construction:")
        for k, v in bad_dev.items():
            print(f"      {k:<40s} {v:+.2f}")
    else:
        print("    In Development reconciles on every row")
    if bad_tot:
        print("    Total row != sum of data rows:")
        for k, v in bad_tot.items():
            print(f"      {k:<40s} {v:+.2f}")
    else:
        print("    Total row reconciles")

for tab in ["LNG export capacity by country/area", "LNG import capacity by country/area",
            "LNG export capacity by region", "LNG import capacity by region"]:
    audit_published(tab)


# Third check: does the published region tab agree with the published country tab?
COUNTRY_TO_SUBREGION = (lng[["Country/Area", "SubRegion"]].dropna().drop_duplicates()
                           .set_index("Country/Area")["SubRegion"].to_dict())

def audit_region_vs_country(ft):
    reg = published_frame(f"LNG {ft} capacity by region").drop(index="Total", errors="ignore")
    cty = published_frame(f"LNG {ft} capacity by country/area").drop(index="Total", errors="ignore")
    rolled = cty.groupby(cty.index.map(COUNTRY_TO_SUBREGION)).sum()
    shared = [r for r in reg.index if r in rolled.index]
    print(f"\n=== LNG {ft}: region tab vs its own country tab")
    bad = False
    for r in shared:
        d = (reg.loc[r] - rolled.loc[r]).round(3)
        for col, v in d.items():
            if abs(v) > 0.06:
                print(f"    {r:<30s} {col:<42s} {v:+.2f}")
                bad = True
    if not bad:
        print("    every subregion agrees with the sum of its countries")

for ft in ["export", "import"]:
    audit_region_vs_country(ft)


=== LNG export capacity by country/area
    In Development != Proposed + Construction:
      Total                                    -9.60
    Total row != sum of data rows:
      Proposed                                 -2.10
      In Development (Proposed + Construction) -11.70
      Cancelled                                -0.10
      Operating                                -0.10

=== LNG import capacity by country/area
    In Development != Proposed + Construction:
      France                                   +0.20
      Germany                                  +5.00
      Netherlands                              +0.10
      Total                                    -0.60
    Total row != sum of data rows:
      In Development (Proposed + Construction) -5.90
      Cancelled                                -0.30

=== LNG export capacity by region
    In Development != Proposed + Construction:
      Latin America and the Caribbean          -0.10
      Northern America             

## Write the workbook

One sheet per published tab, same order as the published workbook. Output is a gitignored
`.xlsx` — this is a validation artifact, not a deliverable.

In [16]:
TAB_ORDER = [
    ("LNG export capacity by country/area", capacity["LNG export capacity by country/area"], True),
    ("LNG import capacity by country/area", capacity["LNG import capacity by country/area"], True),
    ("LNG export capacity by region", region_tabs["LNG export capacity by region"], False),
    ("LNG import capacity by region", region_tabs["LNG import capacity by region"], False),
    ("LNG export capacity by owner", owners["LNG export capacity by owner"], True),
    ("LNG import capacity by owner", owners["LNG import capacity by owner"], True),
    ("Operating LNG capacity by start year", start_year, True),
    ("LNG export terminal capex by country/area", capex["LNG export terminal capex by country/area"], True),
    ("LNG import terminal capex by country/area", capex["LNG import terminal capex by country/area"], True),
    ("LNG export terminal capex by region", capex["LNG export terminal capex by region"], True),
    ("LNG import terminal capex by region", capex["LNG import terminal capex by region"], True),
]

def sheet_name(tab):
    """Excel forbids / \ ? * [ ] : in sheet names and caps them at 31 chars."""
    safe = re.sub(r"[/\?*\[\]:]", "-", tab)
    return safe[:31]

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as xl:
    for name, frame, with_index in TAB_ORDER:
        frame.to_excel(xl, sheet_name=sheet_name(name), index=with_index)

print(f"wrote {OUTPUT_XLSX} ({len(TAB_ORDER)} sheets)")

wrote GGIT-LNG-summary-sheets-Sep2025-reproduction.xlsx (11 sheets)


## Landing-page stats

The handful of figures that get quoted in the release write-up.

In [17]:
ec = capacity["LNG export capacity by country/area"].loc["Total"]
ic = capacity["LNG import capacity by country/area"].loc["Total"]
print(f"{RELEASE_LABEL} LNG terminals\n")
print(f"  export operating       {ec['Operating']:>9,.1f} mtpa")
print(f"  export in development  {ec[IN_DEV]:>9,.1f} mtpa")
print(f"  import operating       {ic['Operating']:>9,.1f} mtpa")
print(f"  import in development  {ic[IN_DEV]:>9,.1f} mtpa")
print(f"\n  terminals (unique)     {lng['ProjectID'].nunique():>9,}")
print(f"  units                  {len(lng):>9,}")
print(f"  countries/areas        {lng['Country/Area'].nunique():>9,}")

September 2025 LNG terminals

  export operating           495.8 mtpa
  export in development    1,001.4 mtpa
  import operating         1,190.3 mtpa
  import in development      661.2 mtpa

  terminals (unique)           813
  units                      1,199
  countries/areas              111


## Forking this for Q4 2026

1. Copy this folder to `releases/summary-sheets/2026-q4-lng-terminals/`.
2. Repoint `DATA_CSV` at the 2026 release download. Pull it through `gem-db-ops`
   (`python ../../../../gem-db-ops/lng/pull.py`) rather than a Sheets tab — the 2025 CSV
   here came from the pivot workbook only because that is what the September run used.
   Check the column names: the all-fields Postgres export uses `TerminalID`/`UnitID` where
   this download uses `ProjectID`/`UnitID`.
3. Nothing to empty - the terminal-level capacity fallback is a rule, not a list of
   September 2025 deltas, so it carries over as-is. Check its printed output each run:
   it names every terminal it fires on.
4. Drop `REFERENCE_JSON` and the whole validation section, or repoint it at the 2026
   published tabs once they exist.
5. Decide the capex question before the run: either accept the recomputed estimates as the
   new method and say so in the release notes, or recover the old `CostUSDPerMtpa` values
   from a pre-2025 backup of the Sheet backend.
6. Raise three items with the tracker lead:
   - the published capacity tabs do not reconcile against themselves (stale `In Development`
     cells on Germany, Western Europe, Northern America, France, the Netherlands; and `Total`
     rows that do not equal the sum of their own data rows in all four tabs);
   - the start-year tab's title says "Operating" while the data includes idled, mothballed
     and retired;
   - `gem-tracker-constants` says `Idle` where the tracker and the published tables say
     `Idled`.